In [ ]:
# Задание task_03_04_07.
#Задание: На сайте Всемирного банка (WB) в разделе Data доступна экономическая статистика о валовом внутреннем продукте (ВВП) на душу населения в долларах США 1.
#Используя подготовленный CSV-файл, реализуйте: загрузку данных; поиск государства по названию, а также государства с максимальным, минимальным ВВП на душу населения; сохранение данных в новый CSV-файл с фильтром по определенному условию (например, топ-10 государств по объему ВВП на душу населения).
# Выполнила: Михалева Полина Вячеславовна
# Группа: ЦИБ-251

import csv


class NoSuchCountryError(Exception):
    def __init__(self, message):
        super().__init__(message)


class IllegalArgumentError(ValueError):
    pass


def load_data(filename):
    """Загрузить данные ВВП на душу населения из csv-файла 'filename'.

    Если значения для какого-либо государства не известно, строка должна
    быть пропущена и отсутствовать в результате.

    Параметры:
        - filename (str): имя файла.

    Результат:
        - list of dict: [
            {
              - "name": str: название государства;
              - "gdp": float: ВВП на душу населения.
            }
            ...
          ]

    Функция не обрабатывает исключения."""
    data = []
    with open(filename, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row['gdp'] and row['gdp'].strip():
                try:
                    gdp_value = float(row['gdp'])
                    data.append({
                        'name': row['name'],
                        'gdp': gdp_value
                    })
                except ValueError:
                    continue
    return data


def search(data, criteria):
    """Выполнить поиск государства-значения в 'data' по критерию 'criteria'.

    Параметры:
        - data (list of dict): структура данных формата 'load_data()';
        - criteria (string): критерий поиска; допустимые значения:
            - "-max-": государство с максимальным ВВП на душу населения;
            - "-min-": государство с минимальным ВВП на душу населения;
            - "Russian Federation": название государства.

    Результат:
        - dict: {
              "name": str: название государства;
              "gdp": float: ВВП на душу населения.
          }
        или
        - NoSuchCountryError: если такой страны нет.
    """
    if criteria == "-max-":
        if not data:
            raise NoSuchCountryError("Нет данных для поиска максимума")
        max_item = max(data, key=lambda x: x['gdp'])
        return max_item

    elif criteria == "-min-":
        if not data:
            raise NoSuchCountryError("Нет данных для поиска минимума")
        min_item = min(data, key=lambda x: x['gdp'])
        return min_item

    else:
        for item in data:
            if item['name'] == criteria:
                return item
        raise NoSuchCountryError(f"Страна '{criteria}' не найдена в данных")


def save_data(filename, data, criteria):
    """Сохранить данные 'data' в csv-файл 'filename' по критерию 'criteria'.

    Параметры:
        - filename (str): имя файла;
        - data (list of dict): структура данных формата 'load_data()';
        - criteria (string): критерий поиска; допустимые значения:
            - "top=X": первые X государств по ВВП на душу населения
                         (целое число > 0, по убыванию значения);
            - "tail=X": последние X государств по ВВП на душу населения
                          (целое число > 0, по возрастанию значения);
            - "greater=X": список государств с ВВП на душу населения, больше
                           чем X (вещ. число, по убыванию значения);
            - "less=X": список государств с ВВП на душу населения, меньше
                          чем X (вещ. число, по возрастанию значения).

    Исключения:
        - IllegalArgumentError: 'criteria' содержит недопустимое значение.
    """
    try:
        if '=' not in criteria:
            raise ValueError("Отсутствует знак '=' в критерии")

        method, value = criteria.split("=", 1)
        sorted_by_gdp_desc = sorted(data, key=lambda x: x['gdp'], reverse=True)
        sorted_by_gdp_asc = sorted(data, key=lambda x: x['gdp'])

        filtered_data = []

        if method == "top":
            x = int(value)
            if x <= 0:
                raise ValueError("X должно быть > 0")
            filtered_data = sorted_by_gdp_desc[:x]

        elif method == "tail":
            x = int(value)
            if x <= 0:
                raise ValueError("X должно быть > 0")
            filtered_data = sorted_by_gdp_asc[:x]

        elif method == "greater":
            x = float(value)
            filtered_data = [item for item in sorted_by_gdp_desc if item['gdp'] > x]

        elif method == "less":
            x = float(value)
            filtered_data = [item for item in sorted_by_gdp_asc if item['gdp'] < x]

        else:
            raise ValueError(f"Неизвестный метод: {method}")

        with open(filename, 'w', encoding='utf-8', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=['name', 'gdp'])
            writer.writeheader()
            writer.writerows(filtered_data)

    except Exception as err:
        raise IllegalArgumentError(
            "Значение параметра 'criteria' может быть "
            "одним из:\n"
            '- "top=X": первые X государств по ВВП на душу населения'
            ' (целое число > 0, по убыванию значения);\n'
            '- "tail=X": последние X государств по ВВП на душу населения'
            ' (целое число > 0, по возрастанию значения);\n'
            '- "greater=X": список государств с ВВП на душу населения, больше'
            ' чем X (вещ. число, по убыванию значения);\n'
            '- "less=X": список государств с ВВП на душу населения, меньше'
            ' чем X (вещ. число, по возрастанию значения).') from err


if __name__ == "__main__":
    try:
        filename = input("Введите имя файла: ")
        save_filename = input("Введите имя файла для сохранения: ")

        data = load_data(filename)
        print(f"Загружено {len(data)} записей")

        max_country = search(data, "-max-")
        print(f"Страна с максимальным ВВП: {max_country['name']} - {max_country['gdp']:.2f}")

        min_country = search(data, "-min-")
        print(f"Страна с минимальным ВВП: {min_country['name']} - {min_country['gdp']:.2f}")

        russia = search(data, "Russian Federation")
        print(f"Российская Федерация: {russia['gdp']:.2f}")

        save_data(save_filename, data, "top=5")
        print(f"Сохранены топ-5 стран в {save_filename}")


    except FileNotFoundError as e:
        print(f"Ошибка: файл не найден - {e}")
    except NoSuchCountryError as e:
        print(f"Ошибка поиска: {e}")
    except IllegalArgumentError as e:
        print(f"Ошибка в критерии: {e}")
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")